### Parallel chains

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [2]:
# TASK 1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system","You are a movie summerizer chatbot"),
        ("human","Write a summary of {input} movie in brief.")
    ]
)

In [3]:
# TASK 2 [llm]

llm = ChatGroq(model = "qwen/qwen3-32b" , temperature = 0)

In [4]:
# TASK 3 [parser]

str_parser = StrOutputParser()

In [5]:
# TASK 4 [custom runable]

from langchain_core.runnables import RunnableLambda

def dict_maker(text:str) ->dict:
    return {"text":text}

dictionary_maker_runnable = RunnableLambda(dict_maker)

### Parallel Chain 1

In [6]:
# Task 1 [prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","you are a linked in post generator"),
    ("human","create a post for following text for linkedin : {text}")
])

# Task 2 [llm]
llm = ChatGroq(model = "qwen/qwen3-32b", temperature = 0)

# Task 3 [parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

In [7]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

### Parallel Chain 2

In [8]:
def insta_chain(text:dict):
    
    text = text["text"]
    
    # Task 1 [prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","you are a smart insta post generator"),
        ("human","create a post for the following text for instagram : {text}")
    ])
    
    # Task 2 [llm]
    llm = ChatGroq(model = "qwen/qwen3-32b", temperature = 0)
    
    # Task 3 [parser]
    str_parser = StrOutputParser()  
    
    chat_insta = insta_prompt | llm | str_parser
    result = chat_insta.invoke({"text":text})
    
    return result

insta_chain_runnable = RunnableLambda(insta_chain)


### Final Orchestration

In [9]:
final_chain = (
                prompt_template |
                llm |
                str_parser |
                dictionary_maker_runnable |
                RunnableParallel(branches = 
                                 {
                                     "linkedin": chain_linkedin , 
                                     "instagram": insta_chain_runnable
                                 }
                                 )
                )

In [10]:
final_chain.invoke("Marco")

{'branches': {'linkedin': '<think>\nOkay, the user wants a LinkedIn post based on the movie summary I provided. Let me start by understanding the goal here. LinkedIn posts are usually professional but can be personal if they\'re about experiences or insights. The user\'s summary is about the movie "Marco," which is a coming-of-age story about a transgender teen. The post should probably highlight themes like identity, resilience, and acceptance, which are relevant to professional audiences but also touch on social issues.\n\nFirst, I need to make sure the tone is appropriate for LinkedIn. It should be engaging, maybe a bit inspirational, and encourage discussion. The user might want to share this to show their awareness of social issues or to connect with others interested in similar topics. They might also want to highlight the importance of inclusion in the workplace or society.\n\nI should start with a catchy headline or a question to grab attention. Maybe something like "A Powerful

### Chain as Runnable

In [13]:
# Task 1 [Beautify Function]

def beautify(final_response : dict)-> dict:
    linkedin_response = final_response['branches']['linkedin']
    insta_response = final_response['branches']['instagram']
    
    return {"linkedin":linkedin_response , "instagram" : insta_response}

beautify_runnable = RunnableLambda(beautify)

# Task 2 [final Chain]
beautified_chain = final_chain | beautify_runnable

In [14]:
beautified_chain.invoke("pushpa")

{'linkedin': '<think>\nOkay, the user wants a LinkedIn post based on the summary of the movie Pushpa. Let me start by understanding the key points from the summary. The main elements are the movie\'s title, director, lead actor, the protagonist\'s journey, the conflict with the forest department, the antagonist, and the themes like greed and environmental impact.\n\nFirst, I need to make it engaging for LinkedIn. LinkedIn posts often aim to inform, inspire, or spark discussion. Since it\'s about a movie, maybe highlight the story\'s relevance to real-world issues like environmental conservation and corruption. Also, mention the actors and director to appeal to those who follow Indian cinema.\n\nThe user provided a detailed summary, but the LinkedIn post should be concise. I should start with a catchy headline, maybe using emojis to grab attention. Then, briefly outline the plot, focusing on Pushpa\'s transformation and the central conflict. Highlight the themes and the film\'s impact. 